# OffsideX — Phase 2: Zero-Leakage Evaluation & PyTorch Training
## `02_train_model_var.ipynb`

1. Loads `/content/drive/MyDrive/Offside_Detector/v2_var/features_var.npz` (`X: (97, 9)`).
2. Audits and drops `pitch_slope` from classifier inputs to guarantee zero data leakage.
3. Benchmarks regularized classic ML models using Stratified 5-Fold CV.
4. Trains `ResidualOffsideNet` (`tanh` physics skip + bounded MLP + Label Smoothing) and exports `offside_pytorch_model.pt`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

drive.mount('/content/drive', force_remount=True)

V2_DIR = Path('/content/drive/MyDrive/Offside_Detector/v2_var')
DATA_PATH = V2_DIR / 'features_var.npz'

data = np.load(DATA_PATH, allow_pickle=True)
X = data['X'].astype(np.float32)
y = data['y'].astype(np.int64)
names = data['names']
feature_names = list(data['feature_names'])

print(f"X: {X.shape} | y: {y.shape} | Offside={int((y==1).sum())} | Onside={int((y==0).sum())}")

df = pd.DataFrame(X, columns=feature_names)
df['y'] = y

eda_df = pd.DataFrame({
    'Mean (Onside=0)': df[df['y'] == 0][feature_names].mean(),
    'Mean (Offside=1)': df[df['y'] == 1][feature_names].mean(),
    'Corr with y': [df[col].corr(df['y']) for col in feature_names],
})
display(eda_df.round(4))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))

models = {
    'Logistic Regression': LogisticRegression(C=0.3, max_iter=1000, random_state=42),
    'SVM (RBF Kernel)': SVC(C=0.8, kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=4, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, max_depth=2, learning_rate=0.05, random_state=42),
}

raw_acc = accuracy_score(y, (X[:, 1] > 0).astype(int)) * 100
var_acc = accuracy_score(y, (X[:, 0] > 0).astype(int)) * 100

results = [
    {'Model': 'Rule 1: Raw 2D (raw_diff_x > 0)', 'Train Acc': raw_acc, 'Val Acc (OOF)': raw_acc,
     'F1-Score': f1_score(y, (X[:, 1] > 0).astype(int)) * 100, 'ROC-AUC': roc_auc_score(y, X[:, 1]) * 100},
    {'Model': 'Rule 2: VAR Line (var_diff_x > 0)', 'Train Acc': var_acc, 'Val Acc (OOF)': var_acc,
     'F1-Score': f1_score(y, (X[:, 0] > 0).astype(int)) * 100, 'ROC-AUC': roc_auc_score(y, X[:, 0]) * 100},
]
ml_oof_probs = {'VAR Rule': X[:, 0]}

for name, model in models.items():
    oof_preds = np.zeros(len(y), dtype=np.int64)
    oof_probs = np.zeros(len(y), dtype=np.float32)
    tr_accs = []

    for train_idx, val_idx in folds:
        scaler = RobustScaler()
        X_tr = np.clip(scaler.fit_transform(X[train_idx]), -3.5, 3.5)
        X_va = np.clip(scaler.transform(X[val_idx]), -3.5, 3.5)
        model.fit(X_tr, y[train_idx])
        tr_accs.append(accuracy_score(y[train_idx], model.predict(X_tr)) * 100)
        oof_preds[val_idx] = model.predict(X_va)
        oof_probs[val_idx] = model.predict_proba(X_va)[:, 1]

    ml_oof_probs[name] = oof_probs
    results.append({
        'Model': name, 'Train Acc': float(np.mean(tr_accs)),
        'Val Acc (OOF)': accuracy_score(y, oof_preds) * 100,
        'F1-Score': f1_score(y, oof_preds) * 100,
        'ROC-AUC': roc_auc_score(y, oof_probs) * 100,
    })

results_df = pd.DataFrame(results).sort_values('Val Acc (OOF)', ascending=False).reset_index(drop=True)
display(results_df.round(2))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].hist(df[df['y'] == 0]['var_diff_x'], bins=18, alpha=0.65, color='green', label='Onside (0)')
axes[0].hist(df[df['y'] == 1]['var_diff_x'], bins=18, alpha=0.65, color='red', label='Offside (1)')
axes[0].axvline(0.0, color='black', linestyle='--', linewidth=1.5, label='VAR Threshold (0.0)')
axes[0].set_title('Distribution of VAR Margin (var_diff_x)'); axes[0].legend(); axes[0].grid(alpha=0.25)

colors = ['#757575' if 'Rule' in m else '#1976d2' for m in results_df['Model']]
bars = axes[1].barh(results_df['Model'], results_df['Val Acc (OOF)'], color=colors, edgecolor='black', alpha=0.85)
axes[1].axvline(var_acc, color='red', linestyle='--', linewidth=1.5, label=f'VAR Rule ({var_acc:.1f}%)')
axes[1].set_xlim(0, 100); axes[1].invert_yaxis(); axes[1].legend(); axes[1].grid(axis='x', alpha=0.25)
for bar, val in zip(bars, results_df['Val Acc (OOF)']):
    axes[1].text(val + 1.2, bar.get_y() + bar.get_height() / 2, f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
import torch
import torch.nn as nn

# Drop pitch_slope to prevent clip leakage
CLEAN_IDX = [i for i, name in enumerate(feature_names) if name != 'pitch_slope']
CLEAN_NAMES = [feature_names[i] for i in CLEAN_IDX]
X_clean = X[:, CLEAN_IDX]

print(f"Leakage-free matrix: {X_clean.shape} | Features: {CLEAN_NAMES}")

clean_ml_results = []
clean_oof_probs = {'VAR Rule (var_diff_x > 0)': X_clean[:, 0]}

for name, model in models.items():
    oof_preds = np.zeros(len(y), dtype=np.int64)
    oof_probs = np.zeros(len(y), dtype=np.float32)
    for train_idx, val_idx in folds:
        sc = RobustScaler()
        X_tr = np.clip(sc.fit_transform(X_clean[train_idx]), -3.5, 3.5)
        X_va = np.clip(sc.transform(X_clean[val_idx]), -3.5, 3.5)
        model.fit(X_tr, y[train_idx])
        oof_preds[val_idx] = model.predict(X_va)
        oof_probs[val_idx] = model.predict_proba(X_va)[:, 1]
    clean_oof_probs[name] = oof_probs
    clean_ml_results.append({
        'Model': name, 'Val Acc (OOF)': accuracy_score(y, oof_preds) * 100,
        'F1-Score': f1_score(y, oof_preds) * 100, 'ROC-AUC': roc_auc_score(y, oof_probs) * 100,
    })

clean_ml_df = pd.DataFrame(clean_ml_results).sort_values('Val Acc (OOF)', ascending=False).reset_index(drop=True)
display(clean_ml_df.round(2))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(42)


class OffsideNet(nn.Module):
    def __init__(self, in_features=8):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Linear(in_features, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.30))
        self.layer2 = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.20))
        self.output_layer = nn.Linear(16, 1)

    def forward(self, x):
        return self.output_layer(self.layer2(self.layer1(x))).squeeze(-1)


demo_model = OffsideNet(in_features=X_clean.shape[1]).to(device)
n_params = sum(p.numel() for p in demo_model.parameters() if p.requires_grad)
dummy_batch = torch.tensor(X_clean[:16], dtype=torch.float32, device=device)
demo_model.eval()
with torch.no_grad():
    demo_probs = torch.sigmoid(demo_model(dummy_batch))
print(f"OffsideNet params: {n_params} | Device: {device} | Sample probs: {np.round(demo_probs[:5].cpu().numpy(), 3)}")


In [ ]:
from sklearn.preprocessing import StandardScaler

# Retain exact 0.0 VAR margin after scaling
phys_feat = np.tanh(X[:, 0] / 0.05).astype(np.float32)
ctx_feats = np.column_stack([
    np.tanh(X[:, 3] / 0.12),
    X[:, 4],
    np.clip(X[:, 5], 0.5, 2.0),
    X[:, 6],
    X[:, 7],
    np.tanh(X[:, 8] / 0.15),
]).astype(np.float32)


class ResidualOffsideNet(nn.Module):
    def __init__(self, ctx_dim=6):
        super().__init__()
        self.physics_head = nn.Linear(1, 1)
        self.mlp = nn.Sequential(
            nn.Linear(ctx_dim + 1, 16), nn.SiLU(), nn.Dropout(0.35),
            nn.Linear(16, 8), nn.SiLU(), nn.Dropout(0.25),
            nn.Linear(8, 1),
        )
        with torch.no_grad():
            self.physics_head.weight.fill_(2.0)
            self.physics_head.bias.fill_(0.0)
            self.mlp[-1].weight.mul_(0.05)
            self.mlp[-1].bias.zero_()

    def forward(self, phys_x, ctx_x):
        phys_logit = self.physics_head(phys_x)
        full_in = torch.cat([phys_x, ctx_x], dim=1)
        corr_logit = 0.5 * torch.tanh(self.mlp(full_in))  # Cap MLP delta to prevent overfit
        return (phys_logit + corr_logit).squeeze(-1)


EPOCHS = 60
LR = 0.005
WEIGHT_DECAY = 0.15
LABEL_SMOOTH = 0.08  # Dampen gradient spikes from outliers

pt_oof_probs = np.zeros(len(y), dtype=np.float32)
pt_oof_preds = np.zeros(len(y), dtype=np.int64)
fold_tr_loss = np.zeros((len(folds), EPOCHS), dtype=np.float32)
fold_va_loss = np.zeros((len(folds), EPOCHS), dtype=np.float32)
fold_va_acc = np.zeros((len(folds), EPOCHS), dtype=np.float32)

for f_idx, (train_idx, val_idx) in enumerate(folds):
    seed_everything(42 + f_idx)
    ctx_scaler = StandardScaler()  # Fit scaler per fold only
    ctx_tr = ctx_scaler.fit_transform(ctx_feats[train_idx]).astype(np.float32)
    ctx_va = ctx_scaler.transform(ctx_feats[val_idx]).astype(np.float32)

    p_tr_t = torch.tensor(phys_feat[train_idx], dtype=torch.float32, device=device).unsqueeze(1)
    p_va_t = torch.tensor(phys_feat[val_idx], dtype=torch.float32, device=device).unsqueeze(1)
    c_tr_t = torch.tensor(ctx_tr, dtype=torch.float32, device=device)
    c_va_t = torch.tensor(ctx_va, dtype=torch.float32, device=device)

    y_tr_raw = torch.tensor(y[train_idx], dtype=torch.float32, device=device)
    y_tr_smooth = y_tr_raw * (1.0 - 2.0 * LABEL_SMOOTH) + LABEL_SMOOTH
    y_va_t = torch.tensor(y[val_idx], dtype=torch.float32, device=device)

    model = ResidualOffsideNet(ctx_dim=ctx_feats.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        tr_loss = criterion(model(p_tr_t, c_tr_t), y_tr_smooth)
        tr_loss.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            va_logits = model(p_va_t, c_va_t)
            va_loss = criterion(va_logits, y_va_t)
            va_preds = (torch.sigmoid(va_logits) >= 0.5).long()
            va_acc = (va_preds == y_va_t.long()).float().mean().item() * 100

        fold_tr_loss[f_idx, epoch] = tr_loss.item()
        fold_va_loss[f_idx, epoch] = va_loss.item()
        fold_va_acc[f_idx, epoch] = va_acc

    model.eval()
    with torch.no_grad():
        final_probs = torch.sigmoid(model(p_va_t, c_va_t)).cpu().numpy()
    pt_oof_probs[val_idx] = final_probs
    pt_oof_preds[val_idx] = (final_probs >= 0.5).astype(int)
    print(f"Fold {f_idx+1}: Train={fold_tr_loss[f_idx,-1]:.4f} | Val={fold_va_loss[f_idx,-1]:.4f} | Acc={fold_va_acc[f_idx,-1]:.1f}%")

pt_acc = accuracy_score(y, pt_oof_preds) * 100
pt_f1 = f1_score(y, pt_oof_preds) * 100
pt_auc = roc_auc_score(y, pt_oof_probs) * 100
print(f"ResidualOffsideNet OOF -> Acc={pt_acc:.2f}% | F1={pt_f1:.2f}% | AUC={pt_auc:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
ep = np.arange(1, EPOCHS + 1)
axes[0].plot(ep, fold_tr_loss.mean(axis=0), label='Mean Train Loss', color='#1976d2', linewidth=2)
axes[0].plot(ep, fold_va_loss.mean(axis=0), label='Mean Val Loss', color='#d32f2f', linewidth=2)
axes[0].set_title('Training vs Validation Loss'); axes[0].legend(); axes[0].grid(alpha=0.25)

axes[1].plot(ep, fold_va_acc.mean(axis=0), label='PyTorch Val Acc (%)', color='#2e7d32', linewidth=2)
axes[1].axhline(var_acc, color='black', linestyle='--', linewidth=1.4, label=f'VAR Rule ({var_acc:.1f}%)')
axes[1].axhline(raw_acc, color='gray', linestyle=':', linewidth=1.4, label=f'Raw 2D ({raw_acc:.1f}%)')
axes[1].set_title('Validation Accuracy'); axes[1].set_ylim(45, 85); axes[1].legend(); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, classification_report

best_ml_row = clean_ml_df.iloc[0]
final_summary = pd.DataFrame([
    {'Stage / Model': '1. Raw 2D Baseline (No VAR)', 'Accuracy (%)': raw_acc,
     'F1-Score (%)': f1_score(y, (X[:, 1] > 0).astype(int)) * 100, 'ROC-AUC (%)': roc_auc_score(y, X[:, 1]) * 100},
    {'Stage / Model': f"2. Best Classic ML ({best_ml_row['Model']})", 'Accuracy (%)': float(best_ml_row['Val Acc (OOF)']),
     'F1-Score (%)': float(best_ml_row['F1-Score']), 'ROC-AUC (%)': float(best_ml_row['ROC-AUC'])},
    {'Stage / Model': '3. Semi-Automated VAR Geometry (var_diff_x > 0)', 'Accuracy (%)': var_acc,
     'F1-Score (%)': f1_score(y, (X[:, 0] > 0).astype(int)) * 100, 'ROC-AUC (%)': roc_auc_score(y, X[:, 0]) * 100},
    {'Stage / Model': '4. PyTorch ResidualOffsideNet (5-Fold OOF)', 'Accuracy (%)': pt_acc,
     'F1-Score (%)': pt_f1, 'ROC-AUC (%)': pt_auc},
])
display(final_summary.round(2))
print(classification_report(y, pt_oof_preds, target_names=['Onside (0)', 'Offside (1)'], digits=3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
cm = confusion_matrix(y, pt_oof_preds)
im = axes[0].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0].set_title(f'Confusion Matrix (OOF Acc = {pt_acc:.1f}%)', fontweight='bold')
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['Pred: Onside (0)', 'Pred: Offside (1)'])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['True: Onside (0)', 'True: Offside (1)'])
for r in range(2):
    for c in range(2):
        pct = cm[r, c] / cm[r].sum() * 100
        axes[0].text(c, r, f'{cm[r, c]}\n({pct:.1f}%)', ha='center', va='center',
                     color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

for label_txt, scores, col, ls in [
    ('Raw 2D Rule', X[:, 1], '#757575', ':'),
    ('VAR Geometry (var_diff_x)', X[:, 0], '#1976d2', '--'),
    ('PyTorch ResidualOffsideNet', pt_oof_probs, '#2e7d32', '-'),
]:
    fpr, tpr, _ = roc_curve(y, scores)
    axes[1].plot(fpr, tpr, color=col, linestyle=ls, linewidth=2.2, label=f'{label_txt} (AUC={roc_auc_score(y, scores)*100:.1f}%)')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.35, label='Random (50%)')
axes[1].set_title('ROC Curves (All 97 Frames)', fontweight='bold')
axes[1].legend(loc='lower right'); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()

seed_everything(42)
final_scaler = StandardScaler()
ctx_all_scaled = final_scaler.fit_transform(ctx_feats).astype(np.float32)

p_all_t = torch.tensor(phys_feat, dtype=torch.float32, device=device).unsqueeze(1)
c_all_t = torch.tensor(ctx_all_scaled, dtype=torch.float32, device=device)
y_all_s = torch.tensor(y, dtype=torch.float32, device=device) * (1.0 - 2.0 * LABEL_SMOOTH) + LABEL_SMOOTH

final_model = ResidualOffsideNet(ctx_dim=ctx_feats.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for epoch in range(EPOCHS):
    final_model.train()
    optimizer.zero_grad()
    loss = criterion(final_model(p_all_t, c_all_t), y_all_s)
    loss.backward()
    optimizer.step()
    scheduler.step()

MODEL_PATH = V2_DIR / 'offside_pytorch_model.pt'
checkpoint = {
    'model_state_dict': final_model.state_dict(),
    'scaler_mean': final_scaler.mean_.astype(np.float32),
    'scaler_scale': final_scaler.scale_.astype(np.float32),
    'feature_names': feature_names,
    'oof_accuracy': float(pt_acc), 'oof_f1': float(pt_f1), 'oof_auc': float(pt_auc),
}
torch.save(checkpoint, MODEL_PATH)
print(f"Saved model checkpoint: {MODEL_PATH}")


def predict_offside_from_features(raw_feat_vec, checkpoint_path=MODEL_PATH):
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    net = ResidualOffsideNet(ctx_dim=6).to(device)
    net.load_state_dict(ckpt['model_state_dict'])
    net.eval()

    phys_v = np.array([[np.tanh(raw_feat_vec[0] / 0.05)]], dtype=np.float32)
    ctx_v = np.array([[
        np.tanh(raw_feat_vec[3] / 0.12), raw_feat_vec[4], np.clip(raw_feat_vec[5], 0.5, 2.0),
        raw_feat_vec[6], raw_feat_vec[7], np.tanh(raw_feat_vec[8] / 0.15),
    ]], dtype=np.float32)
    ctx_v = (ctx_v - ckpt['scaler_mean']) / (ckpt['scaler_scale'] + 1e-8)

    with torch.no_grad():
        prob = float(torch.sigmoid(net(torch.tensor(phys_v, device=device), torch.tensor(ctx_v, device=device))).item())
    return ('OFFSIDE' if prob >= 0.5 else 'ONSIDE'), prob


for idx in [0, 15, 35, 60, 85]:
    dec, prob = predict_offside_from_features(X[idx])
    truth = 'OFFSIDE' if y[idx] == 1 else 'ONSIDE'
    print(f"{str(names[idx])[:38]:38s} | Truth={truth:7s} | PyTorch={dec:7s} (P={prob*100:5.1f}%)")
